# All of Us — Counterfactual Time-Series Extraction Pipeline

**Purpose:** Extract longitudinal lab/vital measurements from the AoU Registered Tier CDR,
pivoted around an *index event* (first drug exposure) into $(x, z, y)$ tensors
suitable for training the CausalJEPA model.

| Symbol | Meaning | Shape |
|--------|---------|-------|
| $x$ | Context — clinical time-series **before** the intervention (days −365 to −1) | `(T_pre, F)` |
| $z$ | Intervention — categorical drug ID mapped to an integer index | scalar |
| $y$ | Target — clinical time-series **after** the intervention (days +1 to +365) | `(T_post, F)` |

**Environment:** Run inside an All of Us Researcher Workbench Jupyter environment
where `WORKSPACE_CDR` is set to the Registered Tier dataset.

## Step 1 — BigQuery Extraction

We isolate the **first drug exposure** per patient per intervention of interest (Day 0),
then pull all `measurement` rows within a ±365-day window around that index date.

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

dataset = os.environ["WORKSPACE_CDR"]
print(f"Using CDR dataset: {dataset}")

In [ ]:
# ── OMOP concept IDs for the interventions we study ────────────────────
# Extend this dict to add more arms.  Keys become z-indices automatically.
INTERVENTION_CONCEPTS: dict[str, int] = {
    "metformin": 1503297,
    "lisinopril": 435216,
}

concept_id_list = ", ".join(str(v) for v in INTERVENTION_CONCEPTS.values())
concept_to_z = {v: i for i, v in enumerate(INTERVENTION_CONCEPTS.values())}

WINDOW_DAYS = 365

print(f"Interventions: {INTERVENTION_CONCEPTS}")
print(f"Concept → z index: {concept_to_z}")

In [ ]:
query = f"""
WITH InterventionCohort AS (
    -- First exposure date per patient × drug  (= Day 0 / index date)
    SELECT
        person_id,
        drug_concept_id                  AS intervention_id,
        MIN(drug_exposure_start_date)     AS index_date
    FROM `{dataset}.drug_exposure`
    WHERE drug_concept_id IN ({concept_id_list})
    GROUP BY person_id, drug_concept_id
),

LongitudinalLabs AS (
    SELECT
        c.person_id,
        c.intervention_id,
        c.index_date,
        m.measurement_concept_id,
        m.measurement_date,
        m.value_as_number,
        DATE_DIFF(m.measurement_date, c.index_date, DAY) AS days_from_index
    FROM InterventionCohort c
    JOIN `{dataset}.measurement` m
      ON c.person_id = m.person_id
    WHERE m.measurement_date
          BETWEEN DATE_SUB(c.index_date, INTERVAL {WINDOW_DAYS} DAY)
              AND DATE_ADD(c.index_date, INTERVAL {WINDOW_DAYS} DAY)
      AND m.value_as_number IS NOT NULL
)

SELECT * FROM LongitudinalLabs
"""

print("Submitting BigQuery job …")
df_raw = pd.read_gbq(query, dialect="standard")
print(f"Extracted {len(df_raw):,} longitudinal lab records "
      f"across {df_raw['person_id'].nunique():,} patients.")

In [ ]:
df_raw.head(10)

## Step 2 — Build the Feature Vocabulary

Map every `measurement_concept_id` that appears in the extract to a fixed
column index.  This vocabulary is later saved alongside the training shards
so the model and inference code share identical feature alignment.

In [ ]:
all_concept_ids = sorted(df_raw["measurement_concept_id"].unique())
concept_to_col: dict[int, int] = {cid: i for i, cid in enumerate(all_concept_ids)}
num_features = len(concept_to_col)

print(f"Feature vocabulary size (F): {num_features}")
print(f"First 10 concepts → columns: {dict(list(concept_to_col.items())[:10])}")

## Step 3 — Pivot into Per-Patient $(x, z, y)$ Tensors

For every patient we split the flat row data around Day 0:

- **Context $x$:** days −365 … −1 (pre-intervention baseline)
- **Target $y$:** days +1 … +365 (post-intervention outcome)

Each window is pivoted so that rows are individual measurement dates and
columns are clinical features (indexed by `concept_to_col`).  We also
produce an observation mask and a timestamps vector (days-from-index
converted to seconds, matching the JEPA encoder's continuous temporal
positional encoding).

In [ ]:
SECONDS_PER_DAY = 86_400


def _window_to_arrays(
    window_df: pd.DataFrame,
    concept_to_col: dict[int, int],
    num_features: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray] | None:
    """Convert a pre- or post-window DataFrame into (values, mask, timestamps).

    Returns None when the window has no usable observations.
    """
    if window_df.empty:
        return None

    unique_days = sorted(window_df["days_from_index"].unique())
    day_to_step = {d: i for i, d in enumerate(unique_days)}
    T = len(unique_days)

    values = np.zeros((T, num_features), dtype=np.float32)
    mask = np.zeros((T, num_features), dtype=np.float32)
    timestamps = np.zeros(T, dtype=np.float64)

    for _, row in window_df.iterrows():
        t = day_to_step[row["days_from_index"]]
        col = concept_to_col.get(row["measurement_concept_id"])
        if col is None:
            continue
        values[t, col] = row["value_as_number"]
        mask[t, col] = 1.0
        timestamps[t] = row["days_from_index"] * SECONDS_PER_DAY

    return values, mask, timestamps

In [ ]:
patients: list[dict] = []
skipped = 0

for (person_id, intervention_id), group in df_raw.groupby(
    ["person_id", "intervention_id"]
):
    pre_df = group[group["days_from_index"] < 0]
    post_df = group[group["days_from_index"] > 0]

    pre = _window_to_arrays(pre_df, concept_to_col, num_features)
    post = _window_to_arrays(post_df, concept_to_col, num_features)

    if pre is None or post is None:
        skipped += 1
        continue

    pre_values, pre_mask, pre_ts = pre
    post_values, post_mask, post_ts = post

    patients.append({
        "person_id": int(person_id),
        "intervention_z": concept_to_z[int(intervention_id)],
        "pre_values": pre_values,
        "pre_mask": pre_mask,
        "pre_timestamps": pre_ts,
        "post_values": post_values,
        "post_mask": post_mask,
        "post_timestamps": post_ts,
    })

print(f"Formatted {len(patients):,} valid patient-intervention pairs.")
print(f"Skipped {skipped:,} pairs missing pre- or post-window data.")

## Step 4 — Sanity Checks

In [ ]:
if patients:
    sample = patients[0]
    print(f"Patient {sample['person_id']} (z={sample['intervention_z']})")
    print(f"  pre  shape: values {sample['pre_values'].shape}, "
          f"mask {sample['pre_mask'].shape}, ts {sample['pre_timestamps'].shape}")
    print(f"  post shape: values {sample['post_values'].shape}, "
          f"mask {sample['post_mask'].shape}, ts {sample['post_timestamps'].shape}")
    print(f"  pre  observation density: "
          f"{sample['pre_mask'].mean():.2%}")
    print(f"  post observation density: "
          f"{sample['post_mask'].mean():.2%}")

In [ ]:
z_counts = pd.Series([p["intervention_z"] for p in patients]).value_counts().sort_index()
z_label = {v: k for k, v in INTERVENTION_CONCEPTS.items()}
for z_idx, count in z_counts.items():
    label = z_label.get(list(INTERVENTION_CONCEPTS.values())[z_idx], "unknown")
    print(f"  z={z_idx} ({label}): {count:,} patients")

## Step 5 — Export Training Shards

Save everything to disk so the local backend can consume it for CausalJEPA training
without re-running BigQuery.

In [ ]:
OUTPUT_DIR = Path("training_data")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Save the feature vocabulary ────────────────────────────────────────
vocab_path = OUTPUT_DIR / "feature_vocab.json"
with open(vocab_path, "w") as f:
    json.dump({str(k): v for k, v in concept_to_col.items()}, f, indent=2)

# ── Save the intervention mapping ─────────────────────────────────────
interventions_path = OUTPUT_DIR / "intervention_map.json"
with open(interventions_path, "w") as f:
    json.dump({
        "concept_to_z": {str(k): v for k, v in concept_to_z.items()},
        "labels": {str(v): k for k, v in INTERVENTION_CONCEPTS.items()},
    }, f, indent=2)

# ── Save patient tensor data as compressed .npz files ────────────────
npz_path = OUTPUT_DIR / "patient_tensors.npz"
np.savez_compressed(
    npz_path,
    person_ids=np.array([p["person_id"] for p in patients]),
    intervention_z=np.array([p["intervention_z"] for p in patients]),
    # Ragged arrays → object arrays; the DataLoader pads per-batch later
    pre_values=np.array([p["pre_values"] for p in patients], dtype=object),
    pre_mask=np.array([p["pre_mask"] for p in patients], dtype=object),
    pre_timestamps=np.array([p["pre_timestamps"] for p in patients], dtype=object),
    post_values=np.array([p["post_values"] for p in patients], dtype=object),
    post_mask=np.array([p["post_mask"] for p in patients], dtype=object),
    post_timestamps=np.array([p["post_timestamps"] for p in patients], dtype=object),
)

print(f"Saved {len(patients):,} patients to {npz_path}")
print(f"Feature vocab ({num_features} features) → {vocab_path}")
print(f"Intervention map → {interventions_path}")

## Step 6 — Save manifest (for downstream reproducibility)

In [ ]:
manifest = {
    "cohort_id": "aou_metformin_lisinopril_v1",
    "index_definition": "First drug_exposure_start_date per person × drug",
    "pre_window_days": WINDOW_DAYS,
    "post_window_days": WINDOW_DAYS,
    "washout_days": 0,
    "num_interventions": len(INTERVENTION_CONCEPTS),
    "num_features": num_features,
    "num_patients": len(patients),
    "feature_vocab_path": str(vocab_path),
    "intervention_map_path": str(interventions_path),
    "tensor_path": str(npz_path),
    "cdr_dataset": dataset,
}

manifest_path = OUTPUT_DIR / "manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print("Manifest:")
for k, v in manifest.items():
    print(f"  {k}: {v}")

---

Pipeline complete. The `training_data/` directory now contains:

| File | Contents |
|------|----------|
| `patient_tensors.npz` | Per-patient `(pre_values, pre_mask, pre_timestamps, post_*, intervention_z)` arrays |
| `feature_vocab.json` | `measurement_concept_id → column index` mapping |
| `intervention_map.json` | `drug_concept_id → z index` + human-readable labels |
| `manifest.json` | Cohort metadata for reproducibility |

Next step: load these shards into the `CausalJEPA` PyTorch `Dataset` / `DataLoader`
and begin training (see `ml/train.py`).